# Weather Feature Merge Pipeline

**Purpose:** Horizontally merge all 12 individual weather variable parquets
(already veg- and subregion-filtered) into a single wide feature table, then
validate the result with shape checks, date range inspection, and missing-rate
reporting before saving.

**Pipeline Overview:**

| Phase | Steps |
|-------|-------|
| **A. Merge** | Use `SWE.parquet` as the base; iteratively inner-join each remaining weather variable on `(lon, lat, day)`; log shape at each step |
| **B. Validate & Save** | Reload merged parquet; inspect shape, dtypes, date range, and per-column missing rates; append QA results to log |

**Input:** `Clean_Data/Weather_Data/Combined_Weather_Data_w_Veg_SubRegion_Filter/`
— 12 individual variable parquets (one per weather feature)

**Outputs:**
- `Clean_Data/Feature_Data/Weather_Data_Merged.parquet` — wide merged feature table (127M rows × 16 cols)
- `Logs/Merge_Features/merge_weather_data_log.txt` — per-file merge log + QA summary

## 0. Configuration

Centralized path configuration — **edit this cell only** to adapt to your local setup.

In [1]:
import os

# ====================== EDIT THESE PATHS ======================
PROJECT_ROOT = r"E:\zcao\CA_Wildfire"

# Input: veg + subregion filtered weather parquets (12 variables, no SWE)
WEATHER_FILTERED_DIR = os.path.join(
    PROJECT_ROOT, "Clean_Data", "Weather_Data",
    "Combined_Weather_Data_w_Veg_SubRegion_Filter"
)

# SWE lives in its own directory (written by 02_03_Data_Clean_-_Snow.ipynb)
SWE_PATH = os.path.join(PROJECT_ROOT, "Clean_Data", "Snow_Data", "SWE.parquet")

# Output paths
FEATURE_DATA_DIR = os.path.join(PROJECT_ROOT, "Clean_Data", "Feature_Data")
MERGED_FILE      = "Weather_Data_Merged.parquet"

LOG_DIR  = os.path.join(PROJECT_ROOT, "Logs", "Merge_Features")
LOG_FILE = "merge_weather_data_log.txt"

for d in [FEATURE_DATA_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Input weather dir : {WEATHER_FILTERED_DIR}")
print(f"SWE path          : {SWE_PATH}")
print(f"Output dir        : {FEATURE_DATA_DIR}")
print(f"Log dir           : {LOG_DIR}")

# Quick existence checks
for label, path in [("WEATHER_FILTERED_DIR", WEATHER_FILTERED_DIR),
                    ("SWE_PATH",             SWE_PATH)]:
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {label}: {path}")

Input weather dir : E:\zcao\CA_Wildfire\Clean_Data\Weather_Data\Combined_Weather_Data_w_Veg_SubRegion_Filter
SWE path          : E:\zcao\CA_Wildfire\Clean_Data\Snow_Data\SWE.parquet
Output dir        : E:\zcao\CA_Wildfire\Clean_Data\Feature_Data
Log dir           : E:\zcao\CA_Wildfire\Logs\Merge_Features
  [OK] WEATHER_FILTERED_DIR: E:\zcao\CA_Wildfire\Clean_Data\Weather_Data\Combined_Weather_Data_w_Veg_SubRegion_Filter
  [OK] SWE_PATH: E:\zcao\CA_Wildfire\Clean_Data\Snow_Data\SWE.parquet


## 1. Environment Setup

In [2]:
import sys
import gc
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import pyproj
from datetime import datetime
from tqdm import tqdm

pd.set_option('display.max_colwidth', None)
gc.collect()

print(f"Python : {sys.version.split('|')[0].strip()}")
print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")
print(f"pyproj : {pyproj.__version__}")

Python : 3.9.13 (main, Aug 25 2022, 23:51:50) [MSC v.1916 64 bit (AMD64)]
pandas : 2.2.2
numpy  : 1.24.4
pyproj : 3.6.1


---

# Phase A: Merge Weather Variables

**Source layout:**
- `Combined_Weather_Data_w_Veg_SubRegion_Filter/` — 11 weather variable parquets
- `Clean_Data/Snow_Data/SWE.parquet` — SWE, written by `02_03_Data_Clean_-_Snow.ipynb`

`SWE.parquet` is loaded first as the base table because its time column is named
`time` (not `day`) — renamed immediately. All 11 weather files are then inner-joined
on `(lon, lat, day)`. Using `SWE` as the base avoids any confusion between the two
source directories.

## 2. Discover Input Files

List all parquet files in the filtered weather directory and confirm the expected
12 variables are present before starting the merge.

In [3]:
weather_files = sorted([f for f in os.listdir(WEATHER_FILTERED_DIR)
                        if f.endswith('.parquet')])

print(f"Weather files found : {len(weather_files)}")
for f in weather_files:
    print(f"  {f}")

print(f"\nSWE file (separate) : {SWE_PATH}")
print(f"  exists: {os.path.exists(SWE_PATH)}")

Weather files found : 11
  dead_fuel_moisture_1000h.parquet
  dead_fuel_moisture_100h.parquet
  max_air_temperature.parquet
  max_relative_humidity.parquet
  min_air_temperature.parquet
  min_relative_humidity.parquet
  precipitation_amount.parquet
  specific_humidity.parquet
  surface_downwelling_shortwave_flux.parquet
  wind_from_direction.parquet
  wind_speed.parquet

SWE file (separate) : E:\zcao\CA_Wildfire\Clean_Data\Snow_Data\SWE.parquet
  exists: True


## 3. Initialise Log & Load Base Table (`SWE`)

The `SWE` file is loaded first and its `time` column renamed to `day` to match
the naming convention of all other weather files. The `year` column (if present)
is retained at this stage for later grouped QA.

In [4]:
log_messages = []
log_messages.append("Task: Merge weather data after veg and subregion filtering")
log_messages.append(f"Processing started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log_messages.append(f"Weather input dir  : {WEATHER_FILTERED_DIR}")
log_messages.append(f"SWE input path     : {SWE_PATH}")
log_messages.append(f"Weather files      : {len(weather_files)}")
log_messages.append("")

# Load SWE as the base table
# Note: SWE uses 'time' column; rename to 'day' to match all weather files
all_features = pd.read_parquet(SWE_PATH)
if 'time' in all_features.columns:
    all_features = all_features.rename(columns={'time': 'day'})

print(f"Base table (SWE) shape : {all_features.shape}")
print(f"Columns                : {list(all_features.columns)}")
log_messages.append(f"Loaded base (SWE) from {SWE_PATH}: {all_features.shape}")

Base table (SWE) shape : (141953628, 5)
Columns                : ['day', 'lat', 'lon', 'SWE', 'year']


## 4. Iterative Inner-Join

Each of the remaining 11 files is joined on the shared keys `(lon, lat, day)`.
The `year` column is dropped from incoming files before joining (it is already
present in the base table). Shape is logged after every merge to track any
unexpected row loss.

> **Expected behaviour:** Row count should be stable or decrease only slightly
> with each join — large drops indicate a grid or date alignment issue.

In [5]:
for file in tqdm(weather_files, desc="Merging weather files"):
    df = pd.read_parquet(os.path.join(WEATHER_FILTERED_DIR, file))

    # Drop 'year' if present — already in base
    if 'year' in df.columns:
        df = df.drop(columns=['year'])

    rows_before  = len(all_features)
    all_features = pd.merge(all_features, df, on=['lon', 'lat', 'day'], how='inner')
    rows_after   = len(all_features)

    msg = (f"{file:<55} in: {df.shape[0]:>12,}  "
           f"merged shape: {str(all_features.shape):>20}  "
           f"row drop: {rows_before - rows_after:>8,}")
    log_messages.append(msg)

    del df
    gc.collect()

print(f"\nFinal merged shape: {all_features.shape}")

Merging weather files: 100%|██████████| 11/11 [30:16<00:00, 165.14s/it]


Final merged shape: (127478960, 16)


## 5. Inspect Column Types

Confirm the expected 16 columns are present and their dtypes are correct
before writing to disk.

In [6]:
print(f"Shape  : {all_features.shape}")
print(f"\nColumn dtypes:")
print(all_features.dtypes.to_string())

Shape  : (127478960, 16)

Column dtypes:
day                                          datetime64[ns]
lat                                                 float64
lon                                                 float64
SWE                                                 float32
year                                                  int32
dead_fuel_moisture_1000hr                           float64
dead_fuel_moisture_100hr                            float64
max_air_temperature                                 float64
max_relative_humidity                               float64
min_air_temperature                                 float64
min_relative_humidity                               float64
precipitation_amount                                float64
specific_humidity                                   float64
surface_downwelling_shortwave_flux_in_air           float64
wind_from_direction                                 float32
wind_speed                                          float64

## 6. Save Merged Feature Table

Write the wide feature table to Parquet, then free memory before the
validation phase.

In [7]:
merged_path = os.path.join(FEATURE_DATA_DIR, MERGED_FILE)

all_features.to_parquet(merged_path, index=False)

print(f"Saved -> {merged_path}")
print(f"File size : {os.path.getsize(merged_path) / 1e9:.2f} GB")

log_messages.append("")
log_messages.append(f"Saved merged parquet -> {merged_path}")

del all_features
gc.collect()

Saved -> E:\zcao\CA_Wildfire\Clean_Data\Feature_Data\Weather_Data_Merged.parquet
File size : 1.52 GB


0

---

# Phase B: Validate Merged Dataset

Reload the saved parquet from disk to confirm it round-trips correctly,
then run shape, date range, and per-column missing-rate checks.
All results are appended to the log.

## 7. Reload & Shape Check

Re-read the merged parquet and verify the row count, column count, and
dtype schema match expectations from Phase A.

In [8]:
gc.collect()

all_features = pd.read_parquet(merged_path)

print(f"Shape  : {all_features.shape}")
print(f"\nColumn dtypes:")
print(all_features.dtypes.to_string())

log_messages.append("")
log_messages.append("=" * 50)
log_messages.append("QA: Reload validation")
log_messages.append(f"  Shape: {all_features.shape}")

Shape  : (127478960, 16)

Column dtypes:
day                                          datetime64[ns]
lat                                                 float64
lon                                                 float64
SWE                                                 float32
year                                                  int32
dead_fuel_moisture_1000hr                           float64
dead_fuel_moisture_100hr                            float64
max_air_temperature                                 float64
max_relative_humidity                               float64
min_air_temperature                                 float64
min_relative_humidity                               float64
precipitation_amount                                float64
specific_humidity                                   float64
surface_downwelling_shortwave_flux_in_air           float64
wind_from_direction                                 float32
wind_speed                                          float64

## 8. Date Range Check

Confirm the dataset spans the expected 1994–2020 period with no
unexpected truncation from the inner joins.

In [9]:
day_min = all_features['day'].min()
day_max = all_features['day'].max()

print(f"Earliest date : {day_min.strftime('%Y-%m-%d')}")
print(f"Latest date   : {day_max.strftime('%Y-%m-%d')}")
print(f"Date range    : {(day_max - day_min).days:,} days")

log_messages.append(f"  Date range: {day_min.strftime('%Y-%m-%d')} to {day_max.strftime('%Y-%m-%d')}")

Earliest date : 1994-01-01
Latest date   : 2020-09-30
Date range    : 9,769 days


## 9. Missing Rate by Column

Check the fraction of `NaN` values per column. Key things to flag:
- **`SWE`**: high missing rate is expected (no snow outside winter/mountain cells)
- **`precipitation_amount`**: ~57% missing is expected (zero-precipitation days
  are excluded from the extended-period source data)
- All other variables should be < 1% missing

In [10]:
missing_rates = all_features.isnull().mean().mul(100).rename('missing_%')

print("Missing rate per column (%):\n")
print(missing_rates.to_string())

log_messages.append("")
log_messages.append("Missing rates (%)")
for col, rate in missing_rates.items():
    log_messages.append(f"  {col:<50} {rate:.4f}%")

Missing rate per column (%):

day                                           0.000000
lat                                           0.000000
lon                                           0.000000
SWE                                           1.709074
year                                          0.000000
dead_fuel_moisture_1000hr                     0.167830
dead_fuel_moisture_100hr                      0.167830
max_air_temperature                           0.116906
max_relative_humidity                         0.167830
min_air_temperature                           0.116906
min_relative_humidity                         0.167832
precipitation_amount                         57.369750
specific_humidity                             0.167830
surface_downwelling_shortwave_flux_in_air     0.167830
wind_from_direction                           0.267880
wind_speed                                    0.167830


## 10. Save Processing Log

Append the pipeline finish timestamp and write the full log to disk.

In [11]:
log_messages.append("")
log_messages.append(f"Pipeline finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

log_path = os.path.join(LOG_DIR, LOG_FILE)
with open(log_path, 'w') as f:
    f.write('\n'.join(log_messages))

print(f"Log saved to: {log_path}")
print()
print('\n'.join(log_messages[-15:]))   # preview last 15 lines

Log saved to: E:\zcao\CA_Wildfire\Logs\Merge_Features\merge_weather_data_log.txt

  SWE                                                1.7091%
  year                                               0.0000%
  dead_fuel_moisture_1000hr                          0.1678%
  dead_fuel_moisture_100hr                           0.1678%
  max_air_temperature                                0.1169%
  max_relative_humidity                              0.1678%
  min_air_temperature                                0.1169%
  min_relative_humidity                              0.1678%
  precipitation_amount                               57.3698%
  specific_humidity                                  0.1678%
  surface_downwelling_shortwave_flux_in_air          0.1678%
  wind_from_direction                                0.2679%
  wind_speed                                         0.1678%

Pipeline finished: 2026-03-17 15:23:41


## 11. Summary

| Phase | Step | Description | Key Result |
|-------|------|-------------|------------|
| A | Discover | List parquets in `Combined_Weather_Data_w_Veg_SubRegion_Filter/` | 12 files |
| A | Load Base | Read `SWE.parquet`, rename `time` → `day` | Base table established |
| A | Merge Loop | Inner-join 11 remaining files on `(lon, lat, day)` | 127M rows × 16 cols |
| A | Log | Record per-file shape and row-drop counts | Audit trail |
| A | Save | `Weather_Data_Merged.parquet` | ~GB-scale output |
| B | Reload | Re-read parquet from disk | Round-trip confirmed |
| B | Date Range | Check min/max `day` | 1994-01-01 → 2020-09-30 |
| B | Missing Rates | Per-column `NaN` % | SWE ~1.7%, precip ~57% (expected) |
| B | Log | Append QA results, save log file | `merge_weather_data_log.txt` |